# Activation Steering with Linear Probe

Re-runs a random sample of 50 problems (25 originally correct + 25 originally failed)
from `traces_all.h5` with probe-guided activation steering.

This split lets us measure both:
- **Recovery** on the 25 failures: does steering flip any of them to correct?
- **Robustness** on the 25 successes: does the probe avoid messing up easy problems
  where the model was already getting it right?

## Method

1. Compute steering vector `v = mean(h+) - mean(h-)` from all collected hidden states in `traces_all.h5`.
2. Load the fine-tuned linear probe (`linear_probe.pt`).
3. For each sampled problem, generate a new response with a hook on `model.model.norm`
   (the post-RMSNorm hidden state, equivalent to `outputs.hidden_states[-1]`):
   - At every paragraph break (`\n\n`) in the CoT, run the probe on the current hidden state.
   - If `P(correct) < STEER_THRESHOLD` (default 0.35), add `v × ALPHA` to the hidden state before it feeds into the LM head.
4. Report per-problem token counts and accuracy, broken down by original label.

## Steering vector intuition

If `h+ ≈ h- + c`, then `v ≈ c` — the exact translation from failing to succeeding activations.
Adding `v × α` to a low-confidence hidden state moves it geometrically toward where the model's
activations live when it solves problems correctly.

**Why steer at the post-norm hidden state.** The vector `v` lives in the representational
space of `outputs.hidden_states[-1]` (post final RMSNorm — that's where the traces were
collected and the probe was trained). Applying it pre-norm or at earlier layers would mix
geometries — `v` would be the wrong direction in the wrong scale. To steer elsewhere, we'd
need to recollect hidden states there and compute a separate `v`.

**Alpha calibration** (linear, exact at this layer):
- `α = 0.3` → logit shift ≈ 0.8 → bumps P(correct) from 0.35 → ~0.55
- `α = 1.0` → logit shift ≈ 2.7 → bumps P(correct) from 0.35 → ~0.89
- Default `ALPHA = 0.5` is a mild-to-moderate push.

The actual logit shift for your probe + `v` is printed in the Load-probe cell — trust that
over the table above.

Adjust `ALPHA` (and `STEER_THRESHOLD`) in the Config cell.

In [13]:
# === Colab bootstrap ===
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/Avni2000/cs639-final-project.git"
    REPO_DIR = "/content/cs639-final-project"
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull
    %cd {REPO_DIR}/probe-baseline

    OUTPUT_DIR = "/content/drive/MyDrive/cs639-outputs"
else:
    OUTPUT_DIR = "."

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"IN_COLAB={IN_COLAB}  OUTPUT_DIR={OUTPUT_DIR}")

IN_COLAB=False  OUTPUT_DIR=.


In [14]:
%pip install -q "transformers==4.45.0" "torch>=2.3" "datasets" "pandas" \
               "math-verify[antlr4_13_2]" "tqdm" "h5py"


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Config

In [15]:
# --- Inputs ---
HDF5_PATH   = os.path.join(OUTPUT_DIR, "traces_all.h5")    # from shard_merge.ipynb
PROBE_PATH  = os.path.join(OUTPUT_DIR, "linear_probe.pt")  # from probe_train.ipynb
CSV_PATH    = "./aime_historical.csv"          # AIME problem set

# --- Outputs ---
STEERING_TRACES_PATH  = os.path.join(OUTPUT_DIR, "steering_traces.h5")
STEERING_SUMMARY_PATH = os.path.join(OUTPUT_DIR, "steering_summary.json")

# --- Model ---
MODEL_ID        = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
HIDDEN_DIM      = 3584
PROBE_LAYER     = -1        # last hidden layer (layer 27); read AND write here
MAX_NEW_TOKENS  = 8192

# --- Steering ---
STEER_THRESHOLD = 0.45      # steer when probe confidence < this
ALPHA           = 0.8       # scale factor for v; see calibration notes in intro
MAX_STEERS_PER_PROBLEM = 50 # cap to avoid runaway steering

# --- Sampling ---
N_PER_GROUP = 25            # 25 originally correct + 25 originally failed = 50 problems
SAMPLE_SEED = 0

In [16]:
import torch
assert torch.cuda.is_available(), "GPU required. Runtime -> Change runtime type -> GPU."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: NVIDIA H100 80GB HBM3
VRAM: 85.0 GB


## Load probe and compute steering vector

In [17]:
import torch
import torch.nn as nn
import h5py

assert os.path.exists(HDF5_PATH),  f"Missing {HDF5_PATH} — run shard_merge.ipynb first"
assert os.path.exists(PROBE_PATH), f"Missing {PROBE_PATH} — run probe_train.ipynb first"

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Linear probe ---
class LinearProbe(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        return self.fc(x).squeeze(-1)

ckpt = torch.load(PROBE_PATH, map_location=device)
probe = LinearProbe(ckpt["hidden_dim"]).to(device)
probe.load_state_dict(ckpt["state_dict"])
probe.eval()

mu  = ckpt["norm_mean"].to(device)   # [HIDDEN_DIM]
sig = ckpt["norm_std"].to(device)    # [HIDDEN_DIM]

print(f"Probe loaded (best_val_auc={ckpt['best_val_auc']:.4f})")

# --- Steering vector: v = mean(h+) - mean(h-) over all stored hidden states ---
h_pos_parts, h_neg_parts = [], []
with h5py.File(HDF5_PATH, "r") as f:
    for key in sorted(f.keys()):
        if not key.startswith("problem_"):
            continue
        grp = f[key]
        hs = torch.from_numpy(grp["hidden_states"][...]).float()
        if hs.shape[0] == 0:
            continue
        label = int(grp["label"][()])
        if label == 1:
            h_pos_parts.append(hs)
        else:
            h_neg_parts.append(hs)

H_pos = torch.cat(h_pos_parts, dim=0)  # [N+, HIDDEN_DIM]
H_neg = torch.cat(h_neg_parts, dim=0)  # [N-, HIDDEN_DIM]
steering_vec = (H_pos.mean(0) - H_neg.mean(0)).to(device)  # [HIDDEN_DIM]

print(f"h+: {H_pos.shape}  h-: {H_neg.shape}")
print(f"steering_vec norm: {steering_vec.norm().item():.2f}")

# Sanity: what does alpha * v do to the probe logit?
# In normalized space: shift = w · (v * alpha / sig)
w = ckpt["state_dict"]["fc.weight"].squeeze(0).to(device)
logit_shift = (w * steering_vec.to(w.device) / sig).sum().item() * ALPHA
p_before = torch.sigmoid(torch.tensor(torch.log(torch.tensor(STEER_THRESHOLD / (1 - STEER_THRESHOLD)))))
import math
base_logit = math.log(STEER_THRESHOLD / (1 - STEER_THRESHOLD))
p_after = 1.0 / (1.0 + math.exp(-(base_logit + logit_shift)))
print(f"\nAlpha={ALPHA}: probe logit shifts by {logit_shift:.3f}")
print(f"  P(correct) at threshold: {STEER_THRESHOLD:.2f} → {p_after:.3f} after steering")

Probe loaded (best_val_auc=0.8707)
h+: torch.Size([34869, 3584])  h-: torch.Size([28116, 3584])
steering_vec norm: 95.20

Alpha=0.8: probe logit shifts by 2.174
  P(correct) at threshold: 0.45 → 0.878 after steering


/tmp/ipykernel_298/1077386893.py:56: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  p_before = torch.sigmoid(torch.tensor(torch.log(torch.tensor(STEER_THRESHOLD / (1 - STEER_THRESHOLD)))))


## Sample problems

Randomly draw `N_PER_GROUP` originally-correct and `N_PER_GROUP` originally-failed
problems from the merged HDF5. The originally-correct group is the robustness check;
the originally-failed group is the recovery test.

In [18]:
import random
import pandas as pd

# Load AIME problem set
df = pd.read_csv(CSV_PATH)
problems = df[["Question", "Answer"]].dropna().reset_index(drop=True)

# Bucket problem IDs by original label from the merged HDF5
correct_ids, failed_ids = [], []
with h5py.File(HDF5_PATH, "r") as f:
    for key in sorted(f.keys()):
        if not key.startswith("problem_"):
            continue
        pid = int(key.split("_", 1)[1])
        if pid >= len(problems):
            continue  # outside CSV range
        label = int(f[key]["label"][()])
        (correct_ids if label == 1 else failed_ids).append(pid)

print(f"Available — originally correct: {len(correct_ids)}, originally failed: {len(failed_ids)}")

rng = random.Random(SAMPLE_SEED)
n_c = min(N_PER_GROUP, len(correct_ids))
n_f = min(N_PER_GROUP, len(failed_ids))
sampled_correct = sorted(rng.sample(correct_ids, n_c))
sampled_failed  = sorted(rng.sample(failed_ids,  n_f))

# original_label: 1 if originally correct, 0 if originally failed
sampled = [(pid, 1) for pid in sampled_correct] + [(pid, 0) for pid in sampled_failed]
sampled.sort()

print(f"Sampled — correct: {n_c}, failed: {n_f}, total: {len(sampled)}")
print(f"Correct IDs: {sampled_correct[:10]}{'...' if n_c > 10 else ''}")
print(f"Failed  IDs: {sampled_failed[:10]}{'...' if n_f > 10 else ''}")

Available — originally correct: 387, originally failed: 285
Sampled — correct: 25, failed: 25, total: 50
Correct IDs: [98, 130, 183, 188, 218, 221, 269, 296, 300, 311]...
Failed  IDs: [9, 10, 92, 116, 152, 154, 161, 185, 203, 322]...


## Load model

In [19]:
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print(f"Model loaded in {time.time() - t0:.1f}s")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded in 40.0s


## Helpers

In [20]:
def build_prompt(problem_text: str) -> str:
    messages = [{
        "role": "user",
        "content": (
            "Solve the following AIME problem. "
            "Your final answer must be a non-negative integer (0-999), "
            "placed inside \\boxed{}.\n\n" + problem_text
        ),
    }]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


@torch.no_grad()
def probe_confidence(h_raw: torch.Tensor) -> float:
    """Run probe on a single raw hidden state vector [HIDDEN_DIM]."""
    h = h_raw.to(device).float()
    h_norm = (h - mu) / sig
    logit = probe.fc(h_norm.unsqueeze(0)).squeeze()
    return torch.sigmoid(logit).item()


# Hook target: model.model.norm produces outputs.hidden_states[-1] (post final RMSNorm,
# pre LM head). The probe was trained on this tensor, and the steering vector lives in
# this space. Hooking model.model.layers[-1] would be PRE-norm — different geometry.
def _final_hidden_module():
    return model.model.norm


def generate_with_steering(prompt: str):
    """
    Run greedy generation with probe-gated activation steering.

    At each paragraph break (\\n\\n) inside the CoT:
      - Run probe on the post-RMSNorm hidden state of that token.
      - If P(correct) < STEER_THRESHOLD, add steering_vec * ALPHA to the
        hidden state before it reaches the LM head.

    Returns:
        generated_text (str),
        generated_ids  (np.ndarray int32, [n_tokens]),
        n_steers       (int),
        steer_log      (list of dicts with token_pos, conf_before, conf_after)
    """
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    prompt_len = input_ids.shape[1]

    # Shared mutable state for the hooks
    state = {
        "generated_ids": [],          # token ids generated so far
        "current_embed_ids": None,    # input to embed_tokens at this step
        "is_prompt_pass": True,       # True during the first (prompt) forward pass
        "prev_text": "",              # decoded text seen at last paragraph-check
        "steer_count": 0,
        "steer_log": [],              # (token_position, conf_before, conf_after)
    }

    # Hook 1: capture which token(s) are being processed right now
    def embed_hook(module, input, output):
        ids = input[0][0].tolist()  # [seq_len]
        state["current_embed_ids"] = ids
        # After the prompt pass, each step processes exactly one new token
        if not state["is_prompt_pass"]:
            state["generated_ids"].extend(ids)

    # Hook 2: probe + steer at paragraph breaks (on post-RMSNorm output, which is a
    # plain tensor — not a tuple like decoder-layer outputs).
    def norm_hook(module, input, output):
        if state["is_prompt_pass"]:
            state["is_prompt_pass"] = False
            return output   # don't steer during prompt encoding

        if state["steer_count"] >= MAX_STEERS_PER_PROBLEM:
            return output

        # Decode the CoT generated so far and check for a new \n\n
        current_text = tokenizer.decode(state["generated_ids"], skip_special_tokens=False)
        new_suffix = current_text[len(state["prev_text"]):]

        if "\n\n" not in new_suffix:
            return output

        # Advance prev_text past the first \n\n we found
        para_pos = state["prev_text"] + new_suffix.split("\n\n", 1)[0] + "\n\n"
        state["prev_text"] = para_pos

        # Hidden state for the current (last) token: shape [1, 1, HIDDEN_DIM]
        hs = output  # tensor [batch, seq_len, hidden_dim]
        h_last = hs[0, -1, :].float()  # [hidden_dim]

        conf = probe_confidence(h_last)

        if conf < STEER_THRESHOLD:
            sv = steering_vec.to(hs.device).to(hs.dtype)
            new_hs = hs.clone()
            new_hs[0, -1, :] = hs[0, -1, :] + sv * ALPHA

            conf_after = probe_confidence(new_hs[0, -1, :].float())
            state["steer_count"] += 1
            state["steer_log"].append({
                "token_pos": len(state["generated_ids"]),
                "conf_before": float(conf),
                "conf_after":  float(conf_after),
            })
            return new_hs

        return output

    h_embed = model.model.embed_tokens.register_forward_hook(embed_hook)
    h_norm  = _final_hidden_module().register_forward_hook(norm_hook)

    try:
        with torch.no_grad():
            gen = model.generate(
                input_ids,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
    finally:
        h_embed.remove()
        h_norm.remove()

    generated_ids = gen[0, prompt_len:].cpu().numpy().astype("int32")
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=False)

    return generated_text, generated_ids, state["steer_count"], state["steer_log"]


print("Helpers defined.")

Helpers defined.


## Sanity check: probe layer matches steering layer

The probe was trained on `outputs.hidden_states[-1]`. The steering hook writes into
`model.model.layers[-1]`'s output. If these aren't the same tensor we'd be reading and
writing into the wrong place. One forward pass on a real prompt confirms they match.

In [21]:
_test_pid = sampled[0][0]
_test_prompt = build_prompt(problems.iloc[_test_pid]["Question"])
_test_ids = tokenizer(_test_prompt, return_tensors="pt").input_ids.to(model.device)

_captured = {}
def _capture_hook(module, input, output):
    # model.model.norm output is a tensor (not a tuple)
    _captured["hs"] = output.detach().clone()

_h = _final_hidden_module().register_forward_hook(_capture_hook)
try:
    with torch.no_grad():
        _out = model(_test_ids, output_hidden_states=True, use_cache=False)
finally:
    _h.remove()

_hs_hook   = _captured["hs"][0, -1, :].float()
_hs_output = _out.hidden_states[-1][0, -1, :].float()
_diff = (_hs_hook - _hs_output).abs().max().item()
_conf_hook   = probe_confidence(_hs_hook)
_conf_output = probe_confidence(_hs_output)

print(f"Hidden-state max abs diff (norm hook vs outputs.hidden_states[-1]): {_diff:.2e}")
print(f"Probe conf — hook: {_conf_hook:.4f}  outputs: {_conf_output:.4f}  Δ: {abs(_conf_hook - _conf_output):.2e}")
assert _diff < 1e-3, (
    f"Norm hook output disagrees with outputs.hidden_states[-1] "
    f"(max abs diff {_diff:.2e}). Probe and steering hook would target different tensors."
)
print("OK: probe reads the same tensor steering writes into.")

del _out, _captured, _hs_hook, _hs_output
torch.cuda.empty_cache()

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


Hidden-state max abs diff (norm hook vs outputs.hidden_states[-1]): 0.00e+00
Probe conf — hook: 0.5182  outputs: 0.5182  Δ: 0.00e+00
OK: probe reads the same tensor steering writes into.


## Run steering on sampled problems

In [ ]:
import numpy as np
from math_verify import parse, verify
from tqdm.auto import tqdm


def _open_traces_for_resume(path):
    """
    Open the traces HDF5 in append mode. If it exists, verify alpha/threshold
    match; otherwise refuse so we don't mix incompatible runs into one file.
    Returns the set of problem_idxs already finished.
    """
    if not os.path.exists(path):
        with h5py.File(path, "w") as f:
            f.attrs["alpha"]           = float(ALPHA)
            f.attrs["steer_threshold"] = float(STEER_THRESHOLD)
            f.attrs["model_id"]        = MODEL_ID
            f.attrs["probe_layer"]     = int(PROBE_LAYER)
            f.attrs["sample_seed"]     = int(SAMPLE_SEED)
            f.attrs["max_steers"]      = int(MAX_STEERS_PER_PROBLEM)
        return set()

    with h5py.File(path, "r") as f:
        prev_alpha = float(f.attrs.get("alpha", float("nan")))
        prev_thr   = float(f.attrs.get("steer_threshold", float("nan")))
        if prev_alpha != ALPHA or prev_thr != STEER_THRESHOLD:
            raise RuntimeError(
                f"Existing traces use alpha={prev_alpha}, threshold={prev_thr}, "
                f"current is alpha={ALPHA}, threshold={STEER_THRESHOLD}. "
                f"Move/delete {path} to start a fresh run."
            )
        done = {
            int(k.split("_", 1)[1])
            for k in f.keys()
            if k.startswith("problem_")
        }
    return done


def _write_problem(path, problem_idx, record):
    """Write one problem group and close. Done per-problem so a crash mid-run
    leaves a valid file with all completed problems."""
    with h5py.File(path, "a") as f:
        key = f"problem_{problem_idx}"
        if key in f:
            del f[key]  # overwrite if it somehow exists
        grp = f.create_group(key)

        # Scalars/strings as attrs
        grp.attrs["original_label"] = int(record["original_label"])
        grp.attrs["label_steered"]  = int(record["label_steered"])
        grp.attrs["n_tokens"]       = int(record["n_tokens"])
        grp.attrs["n_steers"]       = int(record["n_steers"])
        grp.attrs["wall_s"]         = float(record["wall_s"])
        grp.attrs["correct_answer"] = str(record["correct_answer"])
        grp.attrs["predicted"]      = str(record["predicted"])

        # Full generation
        grp.create_dataset("gen_text",      data=record["gen_text"])
        grp.create_dataset("gen_token_ids", data=record["gen_token_ids"], compression="gzip")

        # When we steered (token positions + before/after probe confidence)
        grp.create_dataset("steer_token_pos",   data=record["steer_token_pos"],   dtype="int32")
        grp.create_dataset("steer_conf_before", data=record["steer_conf_before"], dtype="float32")
        grp.create_dataset("steer_conf_after",  data=record["steer_conf_after"],  dtype="float32")

        f.flush()


def _running_postfix(path):
    if not os.path.exists(path):
        return {"C": "0/0", "F": "0/0"}
    n_oc = n_of = c_oc = c_of = 0
    with h5py.File(path, "r") as f:
        for k in f.keys():
            if not k.startswith("problem_"):
                continue
            ol = int(f[k].attrs["original_label"])
            ls = int(f[k].attrs["label_steered"])
            if ol == 1:
                n_oc += 1
                c_oc += ls
            else:
                n_of += 1
                c_of += ls
    return {"C": f"{c_oc}/{n_oc}", "F": f"{c_of}/{n_of}"}


# --- Resume ---
done_ids  = _open_traces_for_resume(STEERING_TRACES_PATH)
remaining = [(pid, lbl) for (pid, lbl) in sampled if pid not in done_ids]
print(f"Resuming with {len(done_ids)} problems already in {STEERING_TRACES_PATH}")
print(f"To run: {len(remaining)} / {len(sampled)} problems")

t_start = time.time()
pbar = tqdm(remaining, desc="steering")
pbar.set_postfix(_running_postfix(STEERING_TRACES_PATH))

for problem_idx, original_label in pbar:
    row    = problems.iloc[problem_idx]
    prompt = build_prompt(row["Question"])

    t0 = time.time()
    gen_text, gen_token_ids, n_steers, steer_log = generate_with_steering(prompt)
    wall = time.time() - t0
    n_tokens = int(gen_token_ids.shape[0])

    gold = parse(f"${row['Answer']}$")
    pred = parse(gen_text)
    is_correct = bool(verify(gold, pred)) if pred else False

    record = {
        "original_label":    int(original_label),
        "label_steered":     int(is_correct),
        "n_tokens":          n_tokens,
        "n_steers":          n_steers,
        "wall_s":            round(wall, 2),
        "correct_answer":    str(row["Answer"]),
        "predicted":         str(pred[0]) if pred else "<none>",
        "gen_text":          gen_text,
        "gen_token_ids":     gen_token_ids,
        "steer_token_pos":   np.array([s["token_pos"]   for s in steer_log], dtype="int32"),
        "steer_conf_before": np.array([s["conf_before"] for s in steer_log], dtype="float32"),
        "steer_conf_after":  np.array([s["conf_after"]  for s in steer_log], dtype="float32"),
    }

    _write_problem(STEERING_TRACES_PATH, problem_idx, record)
    pbar.set_postfix(_running_postfix(STEERING_TRACES_PATH))

    tag = "C" if original_label == 1 else "F"
    print(
        f"  idx={problem_idx:4d} orig={tag} "
        f"correct={is_correct} tokens={n_tokens} steers={n_steers} wall={wall:.1f}s"
    )

    # Reduce fragmentation across a long run with hooks holding refs.
    torch.cuda.empty_cache()

print(f"\nTotal wall time this run: {(time.time() - t_start) / 60:.1f} min")
print(f"Per-problem trace file: {STEERING_TRACES_PATH}")

Resuming with 0 problems already in ./steering_traces.h5
To run: 50 / 50 problems


steering:   0%|          | 0/50 [00:00<?, ?it/s]

/usr/local/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


  idx=   9 orig=F correct=False tokens=8192 steers=50 wall=263.9s
  idx=  10 orig=F correct=False tokens=8192 steers=50 wall=265.2s
  idx=  92 orig=F correct=False tokens=8192 steers=27 wall=310.4s


## Aggregate results

In [ ]:
import json
import numpy as np

# Read everything back from HDF5 so this cell works after a kernel restart.
records = []
all_steer_pos    = []
all_steer_before = []
all_steer_after  = []

with h5py.File(STEERING_TRACES_PATH, "r") as f:
    file_alpha = float(f.attrs["alpha"])
    file_thr   = float(f.attrs["steer_threshold"])
    for key in sorted(f.keys()):
        if not key.startswith("problem_"):
            continue
        grp = f[key]
        records.append({
            "problem_idx":    int(key.split("_", 1)[1]),
            "original_label": int(grp.attrs["original_label"]),
            "label_steered":  int(grp.attrs["label_steered"]),
            "n_tokens":       int(grp.attrs["n_tokens"]),
            "n_steers":       int(grp.attrs["n_steers"]),
            "wall_s":         float(grp.attrs["wall_s"]),
            "correct_answer": str(grp.attrs["correct_answer"]),
            "predicted":      str(grp.attrs["predicted"]),
        })
        all_steer_pos.extend(grp["steer_token_pos"][...].tolist())
        all_steer_before.extend(grp["steer_conf_before"][...].tolist())
        all_steer_after.extend(grp["steer_conf_after"][...].tolist())

orig_correct = [r for r in records if r["original_label"] == 1]
orig_failed  = [r for r in records if r["original_label"] == 0]

n_oc = len(orig_correct)
n_of = len(orig_failed)
oc_correct_after = sum(r["label_steered"] for r in orig_correct)
of_correct_after = sum(r["label_steered"] for r in orig_failed)

oc_acc_after   = oc_correct_after / max(n_oc, 1)
oc_regressions = n_oc - oc_correct_after
of_acc_after   = of_correct_after / max(n_of, 1)

n_total          = len(records)
n_correct_total  = oc_correct_after + of_correct_after
acc_steered      = n_correct_total / max(n_total, 1)
acc_original     = n_oc / max(n_total, 1)

token_counts = [r["n_tokens"] for r in records]
steer_counts = [r["n_steers"] for r in records]

summary = {
    "traces_path":              STEERING_TRACES_PATH,
    "n_problems":               n_total,
    "n_orig_correct":           n_oc,
    "n_orig_failed":            n_of,
    "acc_original":             round(acc_original, 4),
    "acc_steered":              round(acc_steered, 4),
    "orig_correct_kept":        oc_correct_after,
    "orig_correct_regressions": oc_regressions,
    "orig_failed_recovered":    of_correct_after,
    "alpha":                    file_alpha,
    "steer_threshold":          file_thr,
    "tokens_mean":              round(float(np.mean(token_counts)), 1),
    "tokens_median":            round(float(np.median(token_counts)), 1),
    "tokens_max":               int(np.max(token_counts)),
    "steers_mean":              round(float(np.mean(steer_counts)), 2),
    "steers_total":             int(np.sum(steer_counts)),
    "per_problem":              records,
}

with open(STEERING_SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print(f"Summary -> {STEERING_SUMMARY_PATH}")
print(f"\nOverall:")
print(f"  Original accuracy: {n_oc}/{n_total}  ({acc_original:.1%})")
print(f"  Steered accuracy:  {n_correct_total}/{n_total}  ({acc_steered:.1%})")
print(f"\nRobustness (originally correct, n={n_oc}):")
print(f"  Still correct after steering: {oc_correct_after}/{n_oc}  ({oc_acc_after:.1%})")
print(f"  Regressions:                  {oc_regressions}")
print(f"\nRecovery (originally failed, n={n_of}):")
print(f"  Flipped to correct: {of_correct_after}/{n_of}  ({of_acc_after:.1%})")
print(f"\nToken usage:")
print(f"  mean:   {summary['tokens_mean']:.0f}")
print(f"  median: {summary['tokens_median']:.0f}")
print(f"  max:    {summary['tokens_max']}")
print(f"\nSteering interventions:")
print(f"  total:        {summary['steers_total']}")
print(f"  mean/problem: {summary['steers_mean']:.2f}")

## Per-steer confidence breakdown

In [ ]:
if all_steer_before:
    confs_before = np.array(all_steer_before)
    confs_after  = np.array(all_steer_after)
    print(f"Confidence before steering: mean={confs_before.mean():.4f}  "
          f"min={confs_before.min():.4f}  max={confs_before.max():.4f}")
    print(f"Confidence after  steering: mean={confs_after.mean():.4f}  "
          f"min={confs_after.min():.4f}  max={confs_after.max():.4f}")

    oc_fired = [r for r in records if r["original_label"] == 1 and r["n_steers"] > 0]
    of_fired = [r for r in records if r["original_label"] == 0 and r["n_steers"] > 0]
    oc_steers_total = sum(r["n_steers"] for r in records if r["original_label"] == 1)
    of_steers_total = sum(r["n_steers"] for r in records if r["original_label"] == 0)

    print(f"\nSteering fired on originally-correct problems:")
    print(f"  {len(oc_fired)}/{n_oc} problems, {oc_steers_total} total interventions")
    print(f"  (ideal: low — probe should be confident here)")
    print(f"Steering fired on originally-failed problems:")
    print(f"  {len(of_fired)}/{n_of} problems, {of_steers_total} total interventions")

    oc_fired_kept      = sum(r["label_steered"] for r in oc_fired)
    of_fired_recovered = sum(r["label_steered"] for r in of_fired)
    print(f"\nAmong originally-correct where steering fired:")
    print(f"  {oc_fired_kept}/{len(oc_fired)} still correct "
          f"({oc_fired_kept / max(len(oc_fired), 1):.1%})")
    print(f"Among originally-failed where steering fired:")
    print(f"  {of_fired_recovered}/{len(of_fired)} now correct "
          f"({of_fired_recovered / max(len(of_fired), 1):.1%})")
else:
    print("No steering interventions occurred. Try lowering STEER_THRESHOLD or increasing ALPHA.")